# Publish all slides by permalink

Geef een Instagram-permalink van een Main-post, bekijk eerst hoe **alle** slides er als Snapchat Story uit komen te zien (in dezelfde volgorde als op Instagram), en post ze pas als je zelf de laatste cel draait.

**Let op:** er is geen privé- of testbestemming binnen de Snapchat Business API (zie eerdere uitleg) — de laatste cel post dus echt en meteen live naar het openbare 433-profiel, als losse opeenvolgende Snaps. Draai die alleen als je dat ook echt wil.

In [1]:
import sys, json
import logging
logging.basicConfig(level=logging.INFO)

from Snapchat_Repost import (
    get_ig_json, FIELDS, LIMIT, extract_media_items,
    download_media, fit_image_to_story, fit_video_to_story,
    encrypt_media, create_media, upload_media, post_story, get_access_token,
)
from config import IG_USER_IDS, SNAPCHAT_PROFILE_ID


def find_post_by_permalink(permalink: str, max_pages: int = 20) -> dict:
    """Instagram's Graph API has no lookup-by-permalink endpoint, so this
    pages through recent media (unbounded by the 7-day queue window) until
    it finds a matching permalink."""
    target = permalink.rstrip("/")
    for uid in IG_USER_IDS:
        next_url = f"/{uid}/media"
        for _ in range(max_pages):
            if not next_url:
                break
            params = None if next_url.startswith("http") else {"fields": FIELDS, "limit": LIMIT}
            data = get_ig_json(next_url, params)
            for it in data.get("data", []) or []:
                if (it.get("permalink") or "").rstrip("/") == target:
                    return it
            next_url = data.get("paging", {}).get("next")
    raise ValueError(f"Post not found (or too old for {max_pages} pages): {permalink}")

INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://ds433kv.vault.azure.net/secrets/SocialsAnalyticsMetaApiToken/?api-version=REDACTED'
Request method: 'GET'
Request headers:
    'Accept': 'application/json'
    'x-ms-client-request-id': '43c290a7-a2be-11f1-b65d-4c496c3027c5'
    'User-Agent': 'azsdk-python-keyvault-secrets/4.8.0 Python/3.11.9 (Windows-10-10.0.26200-SP0)'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Response status: 401
Response headers:
    'Cache-Control': 'no-cache'
    'Pragma': 'no-cache'
    'Content-Type': 'application/json; charset=utf-8'
    'Expires': '-1'
    'x-ms-keyvault-region': 'westeurope'
    'x-ms-client-request-id': '43c290a7-a2be-11f1-b65d-4c496c3027c5'
    'x-ms-request-id': '5cca4792-a2b9-4724-85d0-2a5cbbea0efc'
    'x-ms-keyvault-service-version': '1.9.3576.1'
    'x-ms-keyvault-network-info': 'conn_type=Ipv4;addr=185.127.111.220;act_addr_fam=InterNetwork;'
    'X-Content-Type-

## Stap 1 — permalink opgeven en alle slides ophalen

In [ ]:
PERMALINK = "https://www.instagram.com/p/Dck4aZHAsAo/"  # <-- zet hier de permalink die je wil testen

post = find_post_by_permalink(PERMALINK)
slides = extract_media_items(post)
if not slides:
    raise RuntimeError(
        f"Geen postbare media voor deze post (media_type: {post.get('media_type')}) - "
        "Instagram geeft momenteel geen media_url terug. Dit is een bekend, flaky Meta-bugje "
        "(zie eerdere uitleg) - probeer een andere post, of probeer deze straks nog eens."
    )

print(f"post_id: {post['id']}")
print(f"media_type: {post.get('media_type')}  ({len(slides)} slide(s) total)")
print(f"caption: {(post.get('caption') or '')[:200]}")
for i, s in enumerate(slides, start=1):
    print(f"  slide {i}/{len(slides)}: {s['media_type']}")

In [ ]:
from IPython.display import Image, Video, display

processed_slides = []  # [(processed_bytes, is_video), ...] in Instagram order
for i, s in enumerate(slides, start=1):
    raw = download_media(s["url"])
    is_video = s["media_type"] in ("VIDEO", "REEL")
    processed = fit_video_to_story(raw) if is_video else fit_image_to_story(raw)
    processed_slides.append((processed, is_video))

    print(f"--- slide {i}/{len(slides)} ({'video' if is_video else 'image'}) ---")
    if is_video:
        display(Video(data=processed, embed=True, mimetype="video/mp4"))
    else:
        display(Image(data=processed))

## Stap 2 — publiceren (LIVE, PUBLIEK, niet privé)

Zet `CONFIRM_PUBLISH` pas op `True` als je alle previews hierboven hebt gezien en echt wil posten. Deze cel post dan alle slides, in dezelfde volgorde als op Instagram, als losse opeenvolgende Snaps naar het openbare 433 Snapchat-profiel — er is geen manier om dit alleen naar jezelf of Bastiaan te sturen via deze API.

In [ ]:
CONFIRM_PUBLISH = False  # <-- zet dit bewust op True om echt te posten

if not CONFIRM_PUBLISH:
    raise RuntimeError("CONFIRM_PUBLISH staat op False - zet 'm op True in deze cel en run 'm opnieuw om echt te posten.")

access_token = get_access_token()
posted_media_ids = []

for i, (processed, is_video) in enumerate(processed_slides, start=1):
    print(f"=== slide {i}/{len(processed_slides)} ===")
    ciphertext, key, iv = encrypt_media(processed)

    media_name = f"manual_{post['id']}_{i}.mp4" if is_video else f"manual_{post['id']}_{i}.jpg"
    media = create_media(access_token, "VIDEO" if is_video else "IMAGE", name=media_name, key=key, iv=iv)
    print("CREATE_MEDIA:", json.dumps(media, indent=2))

    upload_media(access_token, media["add_path"], media["finalize_path"], ciphertext)
    print("upload done")

    result = post_story(access_token, media["media_id"])
    print("POST_STORY:", json.dumps(result, indent=2))

    posted_media_ids.append(media["media_id"])

print(f"\nGepost naar public_profile {SNAPCHAT_PROFILE_ID}: {len(posted_media_ids)} slide(s) in volgorde, media_ids={posted_media_ids}")